In [1]:
from nn import init_model, ModelMetrics, sep_labels, batch, train, eval
import numpy as np
from jax import random
import itertools
import jax
from tqdm import tqdm
# jax.config.update('jax_platform_name', 'cpu')
from joblib import Parallel, delayed

In [2]:
n_input = 784
n_output = 10
classes = 10

d_train = sep_labels(np.loadtxt("/home/roman/learning-ml/2802ict-assignments/assignment_02/nn/fashion-mnist_train.csv.gz", delimiter=',', skiprows=1), classes) # (60000, 785) first col is label
d_test = sep_labels(np.loadtxt("/home/roman/learning-ml/2802ict-assignments/assignment_02/nn/fashion-mnist_test.csv.gz", delimiter=',', skiprows=1), classes) # (10000, 785) first col is label

key = random.key(0)

(60000, 10) (60000, 784)
(10000, 10) (10000, 784)


In [3]:
# Define the search space
lr_sizes = [0.01, 0.1, 1.0, 3.0]
batch_sizes = [16, 32, 64, 128, 256]
n_hidden_sizes = [32, 64, 128, 164]

n_epochs = 30

hyperparameter_combinations = list(itertools.product(lr_sizes, batch_sizes, n_hidden_sizes))
print(f"Total combinations to test: {len(hyperparameter_combinations)}")

pbar = tqdm(hyperparameter_combinations)

logged_results = []

for (lr, batch_size, n_hidden) in pbar:
    pbar.set_description(f'testing lr={lr} batch_size={batch_size} n_hidden={n_hidden}')

    model = init_model(n_input, n_hidden, n_output)
    
    metrics_train = ModelMetrics()
    for i in range(n_epochs):
        for mb in batch(d_train, batch_size):
            train(mb, model, metrics_train, lr)
        metrics_train.reset()
    
    metrics_val = ModelMetrics()
    for mb in batch(d_test, batch_size):
        eval(mb, model, metrics_val)

    logged_results.append((lr, batch_size, n_hidden, metrics_val.accuracy.value, metrics_val.loss.value))


Total combinations to test: 80


testing lr=3.0 batch_size=256 n_hidden=164: 100%|██████████| 80/80 [41:15<00:00, 30.94s/it] 


In [5]:
import polars as pl
import numpy as np
import plotly.express as px

# logged_results: list of tuples (lr, batch_size, n_hidden, accuracy, loss)
df = pl.DataFrame(
    {
        "lr": [r[0] for r in logged_results],
        "batch_size": [r[1] for r in logged_results],
        "n_hidden": [r[2] for r in logged_results],
        "accuracy": [r[3] for r in logged_results],
        "loss": [r[4] for r in logged_results],
    }
)

# Add log10(lr) column
df = df.with_columns(
    (pl.col("lr").log10()).alias("lr_log10")
)

# Convert to pandas for Plotly (Plotly doesn't take Polars directly)
pdf = df.to_pandas()

fig = px.scatter_3d(
    pdf,
    x="lr_log10", y="batch_size", z="n_hidden",
    color="accuracy", size="accuracy",
    color_continuous_scale="Viridis",
    labels={"lr_log10": "log10(lr)", "accuracy": "Eval accuracy"},
    hover_data={"lr": True, "batch_size": True, "n_hidden": True,
                "accuracy": ":.4f", "loss": ":.4f"}
)
fig.update_traces(marker=dict(opacity=0.85))
fig.show()